# LAB 10 — Tokenization và answer span alignment


In [1]:
import sys,json,inspect
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from IPython.display import display
LAB_DIR=Path.cwd()/'lab10' if (Path.cwd()/'lab10/qa_model.py').is_file() else Path.cwd()
sys.path.insert(0,str(LAB_DIR.resolve()))
import qa_model as qa
qa.configure()
tokenizer=qa.BertTokenizerFast.from_pretrained(qa.assets(),do_lower_case=True)
data=qa.prepare_data()
print({k:len(v) for k,v in data.items()})

{'train': 3000, 'validation': 300, 'test': 600}


W1116 10:28:40.518000 21492 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


#### 1. Vai trò của LAB 10

LAB 08 dự đoán nhãn sequence từ [CLS]; LAB 09 dự đoán BIO theo token; LAB 10 dùng question + context và hai phân phối vị trí để chọn answer span nằm trong context, không sinh câu trả lời mới.


#### 2. Learning Outcomes

Biểu diễn QA bằng start/end; tokenize cặp câu; hiểu offset, overflow và alignment; fine-tune span head; tính EM/F1; phân tích lỗi boundary, distractor, context dài và giới hạn no-answer.


#### 3. Dataset

SQuAD 1.1: id, context, question, answers[{text,answer_start}]. Subset 3.000 câu hỏi từ official train. Official dev được chia theo context rồi lấy 300 validation và 600 test; test trong bài là holdout tự chia từ dev, không phải test ẩn chính thức. Ba tập không trùng context. Subset/seed cố định để phù hợp CPU.


In [2]:
row=data['train'][0]
display(pd.DataFrame([{'id':row['id'],'question':row['question'],'answer':row['answers'][0]['text'],
 'answer_start':row['answers'][0]['answer_start'],'context':row['context']}]))

,id,question,answer,answer_start,context
0,57276d6add62a815002e9c9f,Who threatened the authority of Japan's centra...,Taira no Masakado,324,"Nominally, sovereignty lay in the emperor but ..."


#### 4. Bài toán Extractive QA

(Question, Context) → P_start(i), P_end(j), i≤j. Start/end token dùng chỉ số inclusive; khi lấy text dùng offset start và offset end exclusive. Hai head chỉ chọn vị trí, không dự đoán một chuỗi từ mới.


#### 5. So sánh ba loại Prediction Head

| Task | Input | Output |
| --- | --- | --- |
| Classification | Text | 1 label |
| NER | Text | 1 label/token |
| Extractive QA | Question + Context | Start + End |

Classification: H→h[CLS]→Linear→class. NER: H→Linear per token→BIO. QA: H→Linear(d,2)→tách start/end logits, mỗi nhánh có shape (B,T). Hai cột Linear tương đương hai lớp Linear(d,1).


#### 6. Calculation Exercise 1 — Start / End

Context tokens, đếm từ 0: `[The, capital, of, France, is, Paris, .]`.

| Answer | Start | End (inclusive) |
| --- | ---: | ---: |
| Paris | 5 | 5 |
| capital of France | 1 | 3 |

Các chỉ số này thuộc context token giả định, chưa có question và special tokens. Trong Python lấy `tokens[start:end+1]`.

#### 7. Calculation Exercise 2 — Start Loss

`P_start(5)=0.60`: L_start=-ln(0.60)=**0.510826**. Đề không cho riêng phân phối end, nên L_end=-ln(P_end(j)); nếu end cũng có xác suất 0.60 thì L_end=0.510826 và loss trung bình vẫn là 0.510826. Không tự coi xác suất end đã được đề cung cấp.


#### 8. QA Loss

L=(L_start+L_end)/2, cross-entropy trên các vị trí input. **Vì sao cần hai prediction?** Start xác định nơi bắt đầu nhưng không cho biết answer dài bao nhiêu. End xác định ranh giới kết thúc; chỉ một vị trí không biểu diễn được span có độ dài thay đổi. Training nhận logits trực tiếp, softmax nằm trong cross-entropy.


In [3]:
print('L_start:',-np.log(.60))
print('Nếu P_end=.60, L:',(-np.log(.60)-np.log(.60))/2)

L_start: 0.5108256237659907
Nếu P_end=.60, L: 0.5108256237659907


#### 9. Prediction Before Experiment

| Prediction | Reason | Expected evidence |
| --- | --- | --- |
| 1. Nhiều keyword giống question nhưng không chứa đáp án có thể làm model nhầm span. | Lexical overlap không xác định quan hệ trả lời; heuristic dễ chọn câu distractor, Transformer cần học tương tác question/context. | EM/F1 theo nhóm lexical overlap và cặp context gốc/thêm câu lặp question. |
| 2. Answer dài thường khó đạt EM hơn. | Phải tìm đúng cả start và end; vẫn có thể nhận F1 một phần nếu bỏ sót vài từ. | EM/F1 các nhóm 1,2–3,4–6,7+ từ. |
| 3. Context thêm thông tin không liên quan có thể giảm performance. | Nhiều span cạnh tranh và số window tăng; answer có thể ra ngoài một window dù còn trong toàn context. | So sánh cùng câu hỏi và gold trên original/distractor; giữ model cố định. |
| 4. Đổi thứ tự câu có thể đổi dự đoán dù answer text còn nguyên. | Absolute position và quan hệ context thay đổi. Giữ answer text chưa chắc giữ nguyên mọi tham chiếu ngữ nghĩa. | Thử đảo câu trên cùng 50 ví dụ còn đầy đủ gold text, so EM/F1 và ghi giới hạn phép biến đổi. |


#### 10. Tokenization của QA

BERT dùng `[CLS] question [SEP] context [SEP]`. input_ids tra vocabulary; attention_mask=1 ở token hợp lệ, 0 ở PAD; token_type_ids=0 ở question, 1 ở context cho model này. Không suy ra mọi Transformer đều cần token_type_ids. Question được giới hạn 48 subword trước khi ghép, giống nhau ở mọi cấu hình.


In [4]:
question='What is the capital of France?'
context='The capital of France is Paris.'
encoded=tokenizer(question,context,return_offsets_mapping=True)
display(pd.DataFrame({'tokens':tokenizer.convert_ids_to_tokens(encoded['input_ids']),
 'input_ids':encoded['input_ids'],'attention_mask':encoded['attention_mask'],
 'token_type_ids':encoded['token_type_ids'],'sequence_ids':encoded.sequence_ids(),
 'offset_mapping':encoded['offset_mapping']}))

,tokens,input_ids,attention_mask,token_type_ids,sequence_ids,offset_mapping
0,[CLS],101,1,0,NaN,"(0, 0)"
1,what,2054,1,0,0.0,"(0, 4)"
2,is,2003,1,0,0.0,"(5, 7)"
3,the,1996,1,0,0.0,"(8, 11)"
4,capital,3007,1,0,0.0,"(12, 19)"
5,of,1997,1,0,0.0,"(20, 22)"
6,france,2605,1,0,0.0,"(23, 29)"
7,?,1029,1,0,0.0,"(29, 30)"
8,[SEP],102,1,0,NaN,"(0, 0)"
9,the,1996,1,1,1.0,"(0, 3)"


#### 11. Offset Mapping

Offset là khoảng ký tự [start,end) trong sequence tương ứng: offset question tính theo question, offset context tính theo context. Phải dùng sequence_ids==1 để không nhầm hai hệ offset; special tokens có sequence_id=None và thường offset (0,0). Word index, WordPiece index và character index là ba hệ chỉ số khác nhau.


#### 12. Calculation Exercise 3 — Character Span

Context: `The capital of France is Paris.`

| Đại lượng | Giá trị |
| --- | --- |
| Ký tự đầu P | 25 |
| Ký tự cuối s | 29 |
| Character span half-open | [25,30) |
| Python slice | context[25:30] = Paris |
| Context-only word/token giả định | Paris, vị trí 5 |

Khi tokenize cặp question/context, token index bao gồm question và special tokens nên phải lấy từ offset_mapping, không giữ nguyên số 5. Token end là inclusive; character end dùng slicing là exclusive.


In [5]:
answer_start=context.index('Paris');answer_end=answer_start+len('Paris')
start_token,end_token=qa.token_span(encoded['offset_mapping'],encoded.sequence_ids(),answer_start,answer_end)
print('Character span:',(answer_start,answer_end),'token span:',(start_token,end_token))
print('Answer recovered:',context[encoded['offset_mapping'][start_token][0]:encoded['offset_mapping'][end_token][1]])

Character span: (25, 30) token span: (14, 14)
Answer recovered: Paris


#### 13. Truncation

Sliding window giữ question và trượt trên context, stride=64 subword overlap; max_length gồm question và special tokens. Mọi window được decode và chọn span có start_logit+end_logit cao nhất toàn câu hỏi. Window không chứa đủ gold được gán CLS=(0,0) khi train. Đây không phải model no-answer đã đánh giá trên SQuAD 2.0. Answer dài hơn overlap có thể không nằm trọn trong một window; mục 14 báo coverage thực tế.


Chọn stride=64 để dùng nhất quán ở cả ba độ dài: với max_length=128, question tối đa 48 và 3 special tokens thì context còn ít nhất 77 vị trí. Stride phải nhỏ hơn ngân sách context; dùng overlap 128 ở cấu hình này sẽ không hợp lệ.
